# 07 — Pruebas BERT sobre dataset_sintetico_v2

Evalúa el clasificador BERT pasando por el **pipeline completo de producción**:
1. Limpieza de texto con spaCy (stopwords, lematización) — igual que en `main.py`
2. Clasificación BERT
3. Override determinista Anexo III
4. Métricas comparativas: con y sin override

**Prerequisito:** haber ejecutado `train.py`

In [ ]:
import os
import sys
from pathlib import Path

# Fix conflicto OpenMP en Windows — antes de import torch
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)

ROOT = Path().resolve().parents[3]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PIPELINE_DIR = ROOT / 'src/classifier/bert_pipeline'
MODEL_DIR    = PIPELINE_DIR / 'models'
BERT_PATH    = MODEL_DIR / 'bert_model'
DATASET      = ROOT / 'src/classifier/dataset_sintetico_v2.csv'

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

if not BERT_PATH.exists():
    raise FileNotFoundError(f'Modelo BERT no encontrado en {BERT_PATH}')
print('Setup OK')

## 1. Cargar modelo BERT y pipeline de producción

In [ ]:
from src.classifier.functions import limpiar_texto
from src.classifier.main import _annex3_override

print(f'Cargando BERT desde: {BERT_PATH}')
tokenizer = AutoTokenizer.from_pretrained(str(BERT_PATH))
model     = AutoModelForSequenceClassification.from_pretrained(str(BERT_PATH))
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

id2label = model.config.id2label
print(f'Device   : {device.upper()}')
print(f'id2label : {id2label}')

print('Pipeline de producción importado OK')

## 2. Cargar dataset

In [ ]:
df = pd.read_csv(DATASET)
print(f'Dataset: {DATASET.name}  |  {len(df)} filas')
print('\nDistribución etiquetas:')
print(df['etiqueta'].value_counts().to_string())

## 3. Inferencia con pipeline completo

Para cada descripción:
1. `limpiar_texto()` — mismo preprocesado que en entrenamiento XGBoost (spaCy)
2. BERT inference sobre el texto limpio
3. `_annex3_override()` — override determinista Anexo III sobre texto original

In [4]:
def bert_predict(text_cleaned: str) -> tuple[str, float, dict]:
    """Inferencia BERT sobre texto ya limpiado."""
    inputs = tokenizer(
        text_cleaned, truncation=True, max_length=256, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    proba    = torch.softmax(logits, dim=-1)[0]
    pred_id  = int(logits.argmax(-1).item())
    label    = id2label[pred_id]
    conf     = float(proba[pred_id].item())
    probs    = {id2label[i]: float(proba[i].item()) for i in range(len(id2label))}
    return label, conf, probs


print('Ejecutando pipeline completo (limpiar → BERT → override)...')
resultados = []

for i, row in df.iterrows():
    texto_original = str(row['descripcion'])

    try:
        # Paso 1: limpieza spaCy (igual que en producción)
        texto_limpio = limpiar_texto(texto_original)

        # Paso 2: clasificación BERT
        risk_level, confidence, probabilities = bert_predict(texto_limpio)

        result_ml = {
            'risk_level'   : risk_level,
            'confidence'   : confidence,
            'probabilities': probabilities,
        }

        # Paso 3: override Anexo III (sobre texto ORIGINAL, igual que en producción)
        result_final = _annex3_override(texto_original, result_ml)

        pred_ml      = result_ml['risk_level']
        pred_final   = result_final['risk_level']
        override     = result_final.get('annex3_override', False)
        override_ref = result_final.get('annex3_ref', '')

    except Exception as e:
        pred_ml = pred_final = 'ERROR'
        confidence = 0.0
        override = False
        override_ref = ''
        print(f'  Error fila {i}: {e}')

    resultados.append({
        'id'              : row.get('id', i),
        'descripcion'     : texto_original,
        'texto_limpio'    : texto_limpio if pred_ml != 'ERROR' else '',
        'etiqueta_real'   : row['etiqueta'],
        'pred_bert'       : pred_ml,
        'pred_final'      : pred_final,
        'confianza'       : round(confidence, 3),
        'override_anexo3' : override,
        'override_ref'    : override_ref,
        'acierto_bert'    : pred_ml == row['etiqueta'],
        'acierto_final'   : pred_final == row['etiqueta'],
    })

    if (i + 1) % 50 == 0:
        print(f'  Procesadas {i + 1}/{len(df)} filas...')

df_res = pd.DataFrame(resultados)
print(f'\nCompletado: {len(df_res)} filas procesadas')
print(f'Overrides Anexo III aplicados: {df_res["override_anexo3"].sum()}')

Ejecutando pipeline completo (limpiar → BERT → override)...
  Procesadas 50/285 filas...
  Procesadas 100/285 filas...
  Procesadas 150/285 filas...
  Procesadas 200/285 filas...
  Procesadas 250/285 filas...

Completado: 285 filas procesadas
Overrides Anexo III aplicados: 2


## 4. Métricas comparativas: BERT puro vs pipeline completo

In [5]:
validas = df_res[df_res['pred_final'] != 'ERROR']
clases  = sorted(validas['etiqueta_real'].unique())

y_true       = validas['etiqueta_real']
y_bert       = validas['pred_bert']
y_final      = validas['pred_final']

f1_bert  = f1_score(y_true, y_bert, average='macro', zero_division=0)
acc_bert = accuracy_score(y_true, y_bert)
f1_final = f1_score(y_true, y_final, average='macro', zero_division=0)
acc_final= accuracy_score(y_true, y_final)

print('='*60)
print(f'BERT puro        — F1 macro: {f1_bert:.4f}  Accuracy: {acc_bert:.4f}')
print(f'+ Override Anexo — F1 macro: {f1_final:.4f}  Accuracy: {acc_final:.4f}')
print(f'Mejora override  : {(f1_final - f1_bert)*100:+.2f}pp F1 macro')
print('='*60)
print()
print('--- Classification Report (pipeline completo) ---')
print(classification_report(y_true, y_final, labels=clases, zero_division=0))

print('--- Aciertos por clase (pipeline completo) ---')
for clase in clases:
    sub  = validas[validas['etiqueta_real'] == clase]
    n_ok = sub['acierto_final'].sum()
    print(f'  {clase:<25} {n_ok:>3}/{len(sub):>3}  ({n_ok/len(sub)*100:.0f}%)')

BERT puro        — F1 macro: 0.6316  Accuracy: 0.6316
+ Override Anexo — F1 macro: 0.6268  Accuracy: 0.6281
Mejora override  : -0.48pp F1 macro

--- Classification Report (pipeline completo) ---
                 precision    recall  f1-score   support

    alto_riesgo       0.69      0.61      0.65        72
    inaceptable       0.82      0.46      0.59        69
riesgo_limitado       0.51      0.79      0.62        72
  riesgo_minimo       0.66      0.64      0.65        72

       accuracy                           0.63       285
      macro avg       0.67      0.63      0.63       285
   weighted avg       0.67      0.63      0.63       285

--- Aciertos por clase (pipeline completo) ---
  alto_riesgo                44/ 72  (61%)
  inaceptable                32/ 69  (46%)
  riesgo_limitado            57/ 72  (79%)
  riesgo_minimo              46/ 72  (64%)


## 5. Impacto del override Anexo III

In [6]:
overrides = validas[validas['override_anexo3']]
if len(overrides) > 0:
    corregidos = overrides[overrides['acierto_final'] & ~overrides['acierto_bert']]
    empeorados = overrides[~overrides['acierto_final'] & overrides['acierto_bert']]
    
    print(f'Overrides totales: {len(overrides)}')
    print(f'  Corrigieron un error BERT: {len(corregidos)}')
    print(f'  Empeoraron una predicción: {len(empeorados)}')
    print()
    print('Referencias legales aplicadas:')
    print(overrides['override_ref'].value_counts().to_string())
    print()
    print('Ejemplos de override:')
    for _, row in overrides.head(4).iterrows():
        estado = '✓ CORRECTO' if row['acierto_final'] else '✗ INCORRECTO'
        print(f'  [{estado}] {row["pred_bert"]} → {row["pred_final"]} ({row["override_ref"]})')
        print(f'    {str(row["descripcion"])[:90]}')
else:
    print('No se aplicaron overrides del Anexo III en este dataset.')

Overrides totales: 2
  Corrigieron un error BERT: 0
  Empeoraron una predicción: 1

Referencias legales aplicadas:
override_ref
Anexo III cat. 5.b    2

Ejemplos de override:
  [✗ INCORRECTO] riesgo_limitado → alto_riesgo (Anexo III cat. 5.b)
    versión beta de sistema que recopila datos genéticos de pacientes hospitalarios y los util
  [✗ INCORRECTO] inaceptable → alto_riesgo (Anexo III cat. 5.b)
    Sistema que recopila datos genéticos de pacientes hospitalarios y los utiliza para discrim


## 6. Matriz de confusión

In [7]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, title in [
    (axes[0], y_bert,  'BERT puro'),
    (axes[1], y_final, 'BERT + Override Anexo III'),
]:
    cm = confusion_matrix(y_true, y_pred, labels=clases)
    ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title}\nF1={f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}', fontsize=11)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Matrices de Confusión — dataset_sintetico_v2', fontsize=13, y=1.02)
plt.tight_layout()
out_path = PIPELINE_DIR / 'models/confusion_matrix_pipeline_completo.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path.name}')

Guardado: confusion_matrix_pipeline_completo.png


C:\Users\rammu\AppData\Local\Temp\ipykernel_10976\3775077107.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Guardar resultados

In [8]:
out_csv = ROOT / 'src/classifier/resultados_datasetv2_pipeline_completo.csv'
df_res.to_csv(out_csv, index=False, encoding='utf-8')
print(f'Resultados guardados: {out_csv.name}')
print()
print('=== RESUMEN FINAL ===')
print(f'Dataset      : {DATASET.name} ({len(df_res)} muestras)')
print(f'BERT puro    : F1={f1_bert:.4f}  Acc={acc_bert:.4f}')
print(f'Con override : F1={f1_final:.4f}  Acc={acc_final:.4f}')
print(f'Overrides    : {df_res["override_anexo3"].sum()}')

Resultados guardados: resultados_datasetv2_pipeline_completo.csv

=== RESUMEN FINAL ===
Dataset      : dataset_sintetico_v2.csv (285 muestras)
BERT puro    : F1=0.6316  Acc=0.6316
Con override : F1=0.6268  Acc=0.6281
Overrides    : 2
